# Part 2 - Byte Pair Encoding from scratch

**Ran Lu**

This notebook reproduces the subword algorithm used in *Neural Machine Translation of Rare
Words with Subword Units* (Sennrich, Haddow, and Birch, 2016). The pair counting, greedy merge
training, encoding, decoding, tests, and evaluation are implemented directly in Python.

## 1. Design choices

I implement the paper's **word-boundary, character-level BPE**, not a library tokenizer:

- text is Unicode case-folded and pre-tokenised into words, numbers, and punctuation;
- every training word starts as characters followed by an explicit `</w>` boundary;
- adjacent-pair frequencies are weighted by word frequency;
- the most frequent pair is merged globally at every iteration;
- ties are broken lexicographically, making training deterministic;
- encoding replays the learned merges in rank order and never merges across `</w>`.

Recomputing all pair counts after each merge is not the most efficient implementation, but it
mirrors the algorithm clearly and is appropriate for a teaching corpus. Production systems
maintain incremental statistics. Also, modern *byte-level* BPE first maps bytes to a fixed
alphabet; the 2016 paper and this reproduction begin with Unicode characters.

In [1]:
from collections import Counter
import re

import pandas as pd

END_WORD = "</w>"
TOKEN_PATTERN = re.compile(
    r"[^\W\d_]+(?:['’][^\W\d_]+)*|\d+(?:[.,]\d+)?|[^\w\s]",
    flags=re.UNICODE,
)


def pretokenize(text):
    return TOKEN_PATTERN.findall(text.casefold())


def normalised_text(text):
    return " ".join(pretokenize(text))

## 2. Core algorithm

Let a vocabulary entry be a tuple of current symbols with a corpus frequency. Pair frequency
is the sum of those word frequencies over every adjacent occurrence. A merge replaces all
non-overlapping occurrences of the selected pair inside each vocabulary entry. Training stops
after the requested number of merges or when no pair reaches `min_frequency`.

In [2]:
def build_word_vocabulary(texts):
    word_counts = Counter(
        token
        for text in texts
        for token in pretokenize(text)
    )
    return Counter({tuple(word) + (END_WORD,): count for word, count in word_counts.items()})


def pair_frequencies(vocabulary):
    counts = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += frequency
    return counts


def merge_sequence(symbols, pair):
    merged = []
    index = 0
    while index < len(symbols):
        if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
            merged.append(symbols[index] + symbols[index + 1])
            index += 2
        else:
            merged.append(symbols[index])
            index += 1
    return tuple(merged)


def merge_vocabulary(vocabulary, pair):
    updated = Counter()
    for symbols, frequency in vocabulary.items():
        updated[merge_sequence(symbols, pair)] += frequency
    return updated


def train_bpe(vocabulary, max_merges, min_frequency=2):
    current = Counter(vocabulary)
    merges, history = [], []

    for step in range(max_merges):
        counts = pair_frequencies(current)
        if not counts:
            break
        best_pair = max(counts, key=lambda pair: (counts[pair], pair))
        best_frequency = counts[best_pair]
        if best_frequency < min_frequency:
            break
        current = merge_vocabulary(current, best_pair)
        merges.append(best_pair)
        history.append({
            "step": step + 1,
            "left": best_pair[0],
            "right": best_pair[1],
            "frequency": best_frequency,
            "symbols after merge": len({symbol for word in current for symbol in word}),
        })
    return merges, current, pd.DataFrame(history)

In [3]:
def encode_word(word, merges):
    symbols = tuple(word.casefold()) + (END_WORD,)
    for pair in merges:
        symbols = merge_sequence(symbols, pair)
    return list(symbols)


def encode_text(text, merges):
    return [
        piece
        for word in pretokenize(text)
        for piece in encode_word(word, merges)
    ]


def decode_pieces(pieces):
    return "".join(pieces).replace(END_WORD, " ").strip()


def display_segmentation(word, merges):
    return " | ".join(piece.replace(END_WORD, "</w>") for piece in encode_word(word, merges))

## 3. Correctness checks on the standard toy vocabulary

The familiar `low / lower / newest / widest` example makes weighted counting inspectable.
`('l', 'o')` occurs seven times: five in `low` and twice in `lower`. The assertions also test
conservation of vocabulary frequency, deterministic training, and exact encode/decode
reconstruction after normalisation.

In [4]:
toy_vocabulary = Counter({
    tuple("low") + (END_WORD,): 5,
    tuple("lower") + (END_WORD,): 2,
    tuple("newest") + (END_WORD,): 6,
    tuple("widest") + (END_WORD,): 3,
})

toy_pairs = pair_frequencies(toy_vocabulary)
assert toy_pairs[("l", "o")] == 7
assert toy_pairs[("e", "s")] == 9

merged_once = merge_vocabulary(toy_vocabulary, ("e", "s"))
assert sum(merged_once.values()) == sum(toy_vocabulary.values())

toy_merges_a, _, toy_history = train_bpe(toy_vocabulary, max_merges=10, min_frequency=2)
toy_merges_b, _, _ = train_bpe(toy_vocabulary, max_merges=10, min_frequency=2)
assert toy_merges_a == toy_merges_b

toy_text = "Lowest, newer!"
assert decode_pieces(encode_text(toy_text, toy_merges_a)) == normalised_text(toy_text)
print("All correctness checks passed.")
toy_history

All correctness checks passed.


,step,left,right,frequency,symbols after merge
0,1,t,</w>,9,11
1,2,s,t</w>,9,10
2,3,e,st</w>,9,10
3,4,o,w,7,10
4,5,l,ow,7,9
5,6,w,est</w>,6,10
6,7,n,e,6,10
7,8,ne,west</w>,6,9
8,9,low,</w>,5,10
9,10,w,i,3,9


## 4. French experiment

The following small corpus is deliberately transparent and stored in the notebook. It covers
recurrent French inflections and vocabulary from NLP. The held-out sentences contain new
word forms, so the comparison exposes the difference between word, character, and subword
representations rather than merely memorising the training lines.

In [5]:
TRAIN_TEXTS = [
    "Les chercheurs analysent des phrases francaises et comparent leurs representations.",
    "La chercheuse analyse une phrase francaise avec une methode reproductible.",
    "Nous construisons des modeles linguistiques pour comprendre les textes.",
    "Le modele apprend les regularites morphologiques dans un grand corpus.",
    "Les modeles apprennent des regularites lexicales et syntaxiques.",
    "Un tokeniseur separe les mots, les nombres et la ponctuation.",
    "La tokenisation en sous-mots limite le nombre de symboles rares.",
    "Les sous-mots reutilisent des fragments frequents entre plusieurs mots.",
    "Une representation par caracteres evite les mots inconnus.",
    "Une representation par mots produit un vocabulaire plus grand.",
    "Le codage compresse progressivement les sequences de caracteres.",
    "Chaque fusion remplace la paire de symboles la plus frequente.",
    "Les frequences sont ponderees par le nombre d'occurrences des mots.",
    "Nous evaluons la segmentation sur des phrases jamais observees.",
    "Les terminaisons verbales partagent souvent des fragments utiles.",
    "Je parle, tu parles, nous parlons et elles parlaient calmement.",
    "Il finit, ils finissent et nous finirons le calcul demain.",
    "Les resultats montrent un compromis entre mots et caracteres.",
    "Ce compromis reduit la longueur sans fermer le vocabulaire.",
    "L'algorithme est deterministe lorsque les egalites sont ordonnees.",
    "La reconstruction verifie que le codage conserve toute l'information.",
    "Les experiences reproductibles documentent chaque choix technique.",
    "La linguistique informatique etudie automatiquement les langues.",
    "Des suffixes similaires apparaissent dans les formes flechies.",
    "Le corpus contient des accents, des apostrophes et de la ponctuation.",
]

HELD_OUT_TEXTS = [
    "Les chercheuses compareront des tokenisations reutilisables.",
    "Une analyse morphologique explique les terminaisons inconnues.",
    "La desambiguisation reduira certaines erreurs linguistiques.",
    "Nous verifierons la reconstruction sur 3 phrases nouvelles.",
    "L'internationalisation produit parfois de longues sequences.",
]

french_vocabulary = build_word_vocabulary(TRAIN_TEXTS)
learned_merges, final_vocabulary, merge_history = train_bpe(
    french_vocabulary, max_merges=200, min_frequency=2
)

print(f"Training word types: {len(french_vocabulary)}")
print(f"Learned merges: {len(learned_merges)}")
merge_history.head(15)

Training word types: 143
Learned merges: 200


,step,left,right,frequency,symbols after merge
0,1,s,</w>,104,30
1,2,e,s</w>,64,31
2,3,e,</w>,55,32
3,4,e,n,37,33
4,5,t,</w>,28,34
5,6,o,n,28,35
6,7,l,es</w>,26,36
7,8,.,</w>,25,36
8,9,r,e,21,37
9,10,t,i,18,38


### Effect of the merge budget

`pieces / word` measures sequence length on held-out text. A larger merge budget should make
sequences shorter but increase the learned symbol inventory. `reversible` verifies that no
information is lost apart from the declared case-folding and pre-tokenisation spaces.

In [6]:
held_out_words = [word for text in HELD_OUT_TEXTS for word in pretokenize(text)]
merge_budgets = sorted(set([0, 20, 50, 100, len(learned_merges)]))
budget_records = []

for budget in merge_budgets:
    active_merges = learned_merges[:budget]
    train_pieces = {
        piece
        for text in TRAIN_TEXTS
        for piece in encode_text(text, active_merges)
    }
    held_out_piece_count = sum(
        len(encode_word(word, active_merges)) for word in held_out_words
    )
    reversible = all(
        decode_pieces(encode_text(text, active_merges)) == normalised_text(text)
        for text in HELD_OUT_TEXTS
    )
    budget_records.append({
        "merges": budget,
        "training symbol types": len(train_pieces),
        "held-out pieces": held_out_piece_count,
        "held-out words": len(held_out_words),
        "pieces / word": held_out_piece_count / len(held_out_words),
        "reversible": reversible,
    })

budget_results = pd.DataFrame(budget_records)
budget_results.style.format({"pieces / word": "{:.2f}"})

,merges,training symbol types,held-out pieces,held-out words,pieces / word,reversible
0,0,29,311,38,8.18,True
1,20,47,217,38,5.71,True
2,50,77,176,38,4.63,True
3,100,121,150,38,3.95,True
4,200,173,127,38,3.34,True


### Comparison with word and character tokenisation

Word tokenisation has the shortest sequences but maps unseen held-out types to an unknown
symbol in a fixed-vocabulary model. Character tokenisation is open-vocabulary over the known
alphabet but produces the longest sequences. BPE should occupy the useful middle ground.

In [7]:
chosen_budget = min(100, len(learned_merges))
chosen_merges = learned_merges[:chosen_budget]
train_words = [word for text in TRAIN_TEXTS for word in pretokenize(text)]
train_word_types = set(train_words)
train_char_types = set("".join(train_words)) | {END_WORD}
held_out_characters = [char for word in held_out_words for char in word]
bpe_piece_count = sum(len(encode_word(word, chosen_merges)) for word in held_out_words)
bpe_types = {
    piece for word in train_words for piece in encode_word(word, chosen_merges)
}

representation_results = pd.DataFrame([
    {
        "representation": "word",
        "training vocabulary": len(train_word_types),
        "held-out units / word": 1.0,
        "held-out unknown rate": sum(word not in train_word_types for word in held_out_words) / len(held_out_words),
    },
    {
        "representation": "character + boundary",
        "training vocabulary": len(train_char_types),
        "held-out units / word": sum(len(word) + 1 for word in held_out_words) / len(held_out_words),
        "held-out unknown rate": sum(char not in train_char_types for char in held_out_characters) / len(held_out_characters),
    },
    {
        "representation": f"BPE ({chosen_budget} merges)",
        "training vocabulary": len(bpe_types),
        "held-out units / word": bpe_piece_count / len(held_out_words),
        "held-out unknown rate": sum(
            piece not in bpe_types
            for word in held_out_words
            for piece in encode_word(word, chosen_merges)
        ) / bpe_piece_count,
    },
])
representation_results.style.format({
    "held-out units / word": "{:.2f}",
    "held-out unknown rate": "{:.1%}",
})

,representation,training vocabulary,held-out units / word,held-out unknown rate
0,word,143,1.00,44.7%
1,character + boundary,29,8.18,0.4%
2,BPE (100 merges),121,3.95,0.7%


The BPE unknown-piece rate above is diagnostic rather than an `<unk>` replacement: this
implementation still emits an unseen Unicode character literally, so decoding remains exact.
Byte-level BPE would guarantee that every emitted atomic unit belongs to its fixed 256-byte
base alphabet.

### Qualitative segmentations

The boundary marker remains attached to the final piece. Shared stems or affixes are useful
when learned, but greedy frequency optimisation does not guarantee linguistically correct
morphemes.

In [8]:
inspection_words = [
    "chercheuses",
    "tokenisations",
    "reutilisables",
    "morphologique",
    "internationalisation",
    "desambiguisation",
]

pd.DataFrame([
    {
        "word": word,
        "BPE segmentation": display_segmentation(word, chosen_merges),
        "seen in training": word in train_word_types,
    }
    for word in inspection_words
])

,word,BPE segmentation,seen in training
0,chercheuses,ch | er | ch | e | u | s | es</w>,False
1,tokenisations,to | k | en | is | ati | ons</w>,False
2,reutilisables,re | u | ti | l | is | a | b | les</w>,False
3,morphologique,mo | r | ph | o | l | o | g | i | que</w>,False
4,internationalisation,in | ter | n | ati | on | al | is | ation</w>,False
5,desambiguisation,de | s | a | m | b | i | gu | is | ation</w>,False


In [9]:
sample = HELD_OUT_TEXTS[0]
sample_pieces = encode_text(sample, chosen_merges)
print("Input:  ", sample)
print("Pieces: ", "  ".join(sample_pieces))
print("Decoded:", decode_pieces(sample_pieces))
assert decode_pieces(sample_pieces) == normalised_text(sample)

Input:   Les chercheuses compareront des tokenisations reutilisables.
Pieces:  les</w>  ch  er  ch  e  u  s  es</w>  com  p  a  re  r  on  t</w>  des</w>  to  k  en  is  ati  ons</w>  re  u  ti  l  is  a  b  les</w>  .</w>
Decoded: les chercheuses compareront des tokenisations reutilisables .


## 5. Conclusion

The implementation reproduces the central BPE mechanism: weighted adjacent-pair counts and
greedy global merges learn a deterministic subword vocabulary. The assertions confirm pair
counting, frequency conservation, deterministic tie handling, and exact reconstruction.

The held-out experiment demonstrates the expected trade-off. Zero merges uses **8.18 units
per word** with 29 training symbol types; 100 merges use **3.95 units per word** with 121
types, and 200 merges reach **3.34 units per word**. Every setting reconstructs the normalised
input exactly. At the chosen 100-merge budget, repeated stems and endings are reused in unseen
French word forms. It avoids the word model's **44.7% unseen-type rate** while producing much
shorter sequences than raw characters.

BPE is frequency-driven rather than morphology-aware: some segments align with stems or
suffixes, while others merely reflect common character sequences. Results also depend on
pre-tokenisation, normalisation, corpus size, and merge budget. For a production tokenizer I
would add byte fallback, special-token handling, streaming pair statistics, and evaluation on
a much larger held-out corpus; these engineering extensions are intentionally separated from
the faithful, inspectable algorithm requested here.

## Reference

Sennrich, R., Haddow, B., & Birch, A. (2016). *Neural Machine Translation of Rare Words with
Subword Units*. Proceedings of ACL 2016. [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)